# Phase 2 — Disease Data Cleaning
### Notebook: `phase_2_disease_clean.ipynb`

**What this notebook does:**
1. Loads all 5 disease CSV files from `data/AirPollutionDeath/`
2. Combines them into a single DataFrame
3. Filters to keep only metric_id 1 (Number) and 3 (Rate) — drops Percent
4. Separates the `India` national aggregate row from state-level rows
5. Adds an `evidence_strength` label to every disease cause
6. Saves `outputs/disease_master.csv` — ready to join with pollution data in Phase 3

---
**Confirmed facts baked into this notebook (verified from Phase 0.5 exploration):**
- All 5 files use standard GBD column names: `location_name`, `cause_name`, `metric_id`, `metric_name`, `year`, `val`, `upper`, `lower`
- All 5 files contain only `measure_id = 1` (Deaths only) — no DALYs anywhere
- All 5 files have 3 metric types: Number (1), Percent (2), Rate (3)
- Folder confirmed: `data/AirPollutionDeath/`
- Row counts confirmed: 1st=6720, 2nd=6720, 3rd=9408, 4th=4032, 5th=1344 → Total=28,224
- All 21 cause name spellings confirmed from Cell 13 of Phase 0.5

---
**Input:** `data/AirPollutionDeath/airpollutiondiseases1st.csv` through `5th.csv`  
**Output:** `outputs/disease_master.csv` + `outputs/disease_national.csv`

---
## Cell 1 — Import Libraries

In [1]:
import pandas as pd
from pathlib import Path        # Clean cross-platform file path handling
from collections import Counter # For counting evidence strength labels

print("Libraries imported successfully")

Libraries imported successfully


---
## Cell 2 — Set Paths

**Confirmed folder from Phase 0.5:** `data/AirPollutionDeath/`  
This is a subfolder inside `data/` — separate from where the station CSVs live.

In [2]:
# ── Folder paths ──────────────────────────────────────────────────────────────
# Path('..') = go up one level from notebooks/ to the project root
BASE_DIR    = Path('..')
DISEASE_DIR = BASE_DIR / 'data' / 'AirPollutionDeath'  # Confirmed from Phase 0.5
OUTPUT_DIR  = BASE_DIR / 'outputs'

# ── Disease file names — confirmed from Phase 0.5 ─────────────────────────────
DISEASE_FILE_NAMES = [
    'airpollutiondiseases1st.csv',   # 6720 rows  — Respiratory infections + TB
    'airpollutiondiseases2nd.csv',   # 6720 rows  — COPD, Asthma, Chronic respiratory
    'airpollutiondiseases3rd.csv',   # 9408 rows  — Cardiovascular diseases
    'airpollutiondiseases4th.csv',   # 4032 rows  — Neonatal causes
    'airpollutiondiseases5th.csv',   # 1344 rows  — Lung cancer
]

# ── Verify the folder and all files exist before doing anything else ──────────
if not DISEASE_DIR.exists():
    raise FileNotFoundError(
        f"Folder not found: {DISEASE_DIR}\n"
        "Make sure you are running this notebook from the notebooks/ folder."
    )

print(f"Disease folder: {DISEASE_DIR}")
print()

all_found = True
for fname in DISEASE_FILE_NAMES:
    exists = (DISEASE_DIR / fname).exists()
    status = "OK" if exists else "MISSING"
    print(f"  [{status}]  {fname}")
    if not exists:
        all_found = False

print()
if all_found:
    print("All 5 disease files confirmed present")
else:
    raise FileNotFoundError("One or more disease files are missing. See above.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder ready: {OUTPUT_DIR}")

Disease folder: ..\data\AirPollutionDeath

  [OK]  airpollutiondiseases1st.csv
  [OK]  airpollutiondiseases2nd.csv
  [OK]  airpollutiondiseases3rd.csv
  [OK]  airpollutiondiseases4th.csv
  [OK]  airpollutiondiseases5th.csv

All 5 disease files confirmed present
Output folder ready: ..\outputs


---
## Cell 3 — Define Evidence Strength Map

Each disease cause gets an `evidence_strength` label based on how firmly science has established the link to air pollution.

**All 21 cause names below are copied exactly from Phase 0.5 Cell 13 output.**  
Python string matching is case-sensitive — any difference in spelling will cause that cause to be flagged `unknown`.

| Label | Meaning |
|---|---|
| `direct` | WHO or IARC confirmed causal link — PM2.5 or Benzene proven to cause this disease |
| `strong` | Pollution severely worsens susceptibility — mechanism well understood |
| `indirect` | Emerging/contested research — PM2.5 in pregnancy linked to neonatal outcomes |

In [3]:
# ── EVIDENCE_STRENGTH_MAP ─────────────────────────────────────────────────────
# Every cause name here is copied EXACTLY from Phase 0.5 Cell 13 output.
# Do not change the spelling — it must match what is in the CSV files.

EVIDENCE_STRENGTH_MAP = {

    # ── FILE 1 — Respiratory infections & TB ── label: 'strong' ──────────────
    # Pollution damages the mucosal lining of airways, making infections far more
    # likely to take hold. The mechanism is well-documented even if the infection
    # is the proximate (immediate) cause of death rather than the pollution itself.
    'Tuberculosis'                                              : 'strong',
    'Lower respiratory infections'                             : 'strong',
    'Upper respiratory infections'                             : 'strong',
    'Otitis media'                                             : 'strong',
    'Respiratory infections and tuberculosis'                  : 'strong',

    # ── FILE 2 — Chronic respiratory diseases ── label: 'direct' ─────────────
    # COPD and Asthma are the most directly pollution-caused diseases.
    # PM2.5 particles (< 2.5 microns diameter) lodge permanently in alveoli,
    # causing irreversible scarring. Asthma attacks are directly triggered by
    # NO2, SO2, and PM10. These links are unambiguous in the medical literature.
    'Pulmonary aspiration and foreign body in airway'          : 'direct',
    'Chronic respiratory diseases'                             : 'direct',
    'Chronic obstructive pulmonary disease'                    : 'direct',
    'Asthma'                                                   : 'direct',
    'Interstitial lung disease and pulmonary sarcoidosis'      : 'direct',

    # ── FILE 3 — Cardiovascular diseases ── label: 'direct' ──────────────────
    # PM2.5 particles are small enough to pass through alveoli into the bloodstream.
    # Once in the blood they cause inflammation, promote clot formation, and damage
    # arterial walls. WHO officially classifies PM2.5 as a cardiovascular risk factor
    # with Class I (highest) evidence level.
    'Cardiovascular diseases'                                  : 'direct',
    'Ischemic heart disease'                                   : 'direct',
    'Stroke'                                                   : 'direct',
    'Ischemic stroke'                                          : 'direct',
    'Hypertensive heart disease'                               : 'direct',
    'Atrial fibrillation and flutter'                          : 'direct',
    'Pulmonary Arterial Hypertension'                          : 'direct',  # Capital A in Arterial — confirmed

    # ── FILE 4 — Neonatal causes ── label: 'indirect' ────────────────────────
    # Research links maternal PM2.5 exposure during pregnancy to preterm birth and
    # other neonatal outcomes. The link exists and is published, but the mechanism
    # is still being studied and the evidence is newer/more contested than for
    # the respiratory and cardiovascular causes above.
    'Neonatal preterm birth'                                   : 'indirect',
    'Neonatal encephalopathy due to birth asphyxia and trauma' : 'indirect',
    'Congenital birth defects'                                 : 'indirect',

    # ── FILE 5 — Lung cancer ── label: 'direct' ──────────────────────────────
    # Benzene and PM2.5 are IARC Group 1 confirmed human carcinogens.
    # This is the most ironclad link in the entire dataset.
    # It connects directly to the Benzene, Toluene, Xylene columns in pollution data.
    'Tracheal, bronchus, and lung cancer'                      : 'direct',
}

print(f"EVIDENCE_STRENGTH_MAP defined — {len(EVIDENCE_STRENGTH_MAP)} causes total")
counts = Counter(EVIDENCE_STRENGTH_MAP.values())
for label in ['direct', 'strong', 'indirect']:
    print(f"   {label:12s}: {counts[label]} causes")

EVIDENCE_STRENGTH_MAP defined — 21 causes total
   direct      : 13 causes
   strong      : 5 causes
   indirect    : 3 causes


---
## Cell 4 — Define State Name Maps

From Phase 0.5 Cell 14, the disease data already uses correct GBD state names  
(e.g. `Jammu & Kashmir and Ladakh`, `Other Union Territories`).  
Phase 1 already mapped pollution state names to match these.  

This map is applied defensively — it corrects any old-spelling variants that might sneak in.

In [4]:
# ── DISEASE_STATE_CLEAN_MAP ───────────────────────────────────────────────────
# Applied to location_name column of disease data.
# Format: 'variant or old spelling' → 'correct GBD standard name'

DISEASE_STATE_CLEAN_MAP = {
    'Jammu and Kashmir'       : 'Jammu & Kashmir and Ladakh',  # Old name before Ladakh added
    'J&K'                     : 'Jammu & Kashmir and Ladakh',  # Abbreviation
    'Orissa'                  : 'Odisha',                      # Renamed in 2011
    'Uttaranchal'             : 'Uttarakhand',                  # Renamed in 2000
    'Pondicherry'             : 'Puducherry',                   # Renamed in 2006
    'Other Union territories' : 'Other Union Territories',     # Case variant
    'other union territories' : 'Other Union Territories',
}

# ── National aggregate names — must be separated from state rows ──────────────
# WHY: 'India' is the national total — the sum of all states.
# Mixing it into state averages would double-count every death in India.
NATIONAL_AGGREGATE_NAMES = {'India', 'South Asia', 'Global'}

print(f"DISEASE_STATE_CLEAN_MAP: {len(DISEASE_STATE_CLEAN_MAP)} variant spellings handled")
print(f"National rows to separate out: {NATIONAL_AGGREGATE_NAMES}")

DISEASE_STATE_CLEAN_MAP: 7 variant spellings handled
National rows to separate out: {'South Asia', 'India', 'Global'}


---
## Cell 5 — Load All 5 Files and Combine

We load each file one by one, tag it with its source file name, then stack all 5.

**Expected total rows: 28,224** (confirmed from Phase 0.5 Cell 13).

All 5 files have identical column structure — stacking them with `pd.concat` is safe.

In [5]:
# ── Expected row counts — confirmed from Phase 0.5 Cell 13 ───────────────────
EXPECTED_ROWS = {
    'airpollutiondiseases1st.csv': 6720,
    'airpollutiondiseases2nd.csv': 6720,
    'airpollutiondiseases3rd.csv': 9408,
    'airpollutiondiseases4th.csv': 4032,
    'airpollutiondiseases5th.csv': 1344,
}

all_dfs = []

for fname in DISEASE_FILE_NAMES:
    df_file = pd.read_csv(DISEASE_DIR / fname)

    # Tag each row with its source file
    # WHY: When 28,000 rows are combined, we can still trace any row back to
    # which of the 5 files it came from — essential if something looks wrong
    df_file['source_file'] = fname

    # Sanity check against known row counts
    expected = EXPECTED_ROWS[fname]
    actual   = len(df_file)
    match    = "OK" if actual == expected else f"WARNING: expected {expected}"
    print(f"  [{match}]  {fname}: {actual:,} rows")

    all_dfs.append(df_file)

# ── Stack all 5 into one DataFrame ───────────────────────────────────────────
# ignore_index=True: resets row numbers 0, 1, 2... instead of keeping per-file numbering
df_disease = pd.concat(all_dfs, axis=0, ignore_index=True)

print()
print(f"Combined shape: {df_disease.shape[0]:,} rows x {df_disease.shape[1]} columns")
print(f"Expected:       28,224 rows")
print()
print("Columns in combined DataFrame:")
for col in df_disease.columns:
    print(f"  '{col}'")

  [OK]  airpollutiondiseases1st.csv: 6,720 rows
  [OK]  airpollutiondiseases2nd.csv: 6,720 rows
  [OK]  airpollutiondiseases3rd.csv: 9,408 rows
  [OK]  airpollutiondiseases4th.csv: 4,032 rows
  [OK]  airpollutiondiseases5th.csv: 1,344 rows

Combined shape: 28,224 rows x 19 columns
Expected:       28,224 rows

Columns in combined DataFrame:
  'population_group_id'
  'population_group_name'
  'measure_id'
  'measure_name'
  'location_id'
  'location_name'
  'sex_id'
  'sex_name'
  'age_id'
  'age_name'
  'cause_id'
  'cause_name'
  'metric_id'
  'metric_name'
  'year'
  'val'
  'upper'
  'lower'
  'source_file'


---
## Cell 6 — Filter: Keep Only Number and Rate Metrics

Each disease × state × year × cause has 3 rows in the raw data:
- **metric_id = 1 → Number**: Raw death count (e.g. 42,000 people died of COPD in UP)
- **metric_id = 2 → Percent**: Share of all deaths — not useful for our pollution correlation
- **metric_id = 3 → Rate**: Deaths per 100,000 population — lets us fairly compare large and small states

**We keep 1 and 3. We drop 2.**  
After this filter, rows drop from 28,224 → approximately 18,816 (exactly two-thirds).

In [6]:
rows_before = len(df_disease)

# ── Keep only metric_id 1 (Number) and 3 (Rate) ───────────────────────────────
# isin([1, 3]) means: keep this row if metric_id equals 1 OR 3
# .copy() creates an independent DataFrame — prevents the SettingWithCopyWarning
# that pandas shows when you modify a slice of a larger DataFrame
df_disease = df_disease[
    df_disease['metric_id'].isin([1, 3])
].copy()

rows_after   = len(df_disease)
rows_dropped = rows_before - rows_after

print(f"Rows before filter : {rows_before:,}")
print(f"Rows dropped       : {rows_dropped:,}  (Percent metric — metric_id=2)")
print(f"Rows remaining     : {rows_after:,}")
print(f"Expected remaining : ~18,816")
print()

# ── Confirm Percent rows are gone ────────────────────────────────────────────
remaining = (
    df_disease[['metric_id', 'metric_name']]
    .drop_duplicates()
    .sort_values('metric_id')
)
print("Metric types remaining:")
print(remaining.to_string(index=False))

if 2 in df_disease['metric_id'].values:
    print("\nWARNING: Percent rows still present — filter may have failed")
else:
    print("\nPercent rows successfully removed")

Rows before filter : 28,224
Rows dropped       : 9,408  (Percent metric — metric_id=2)
Rows remaining     : 18,816
Expected remaining : ~18,816

Metric types remaining:
 metric_id metric_name
         1      Number
         3        Rate

Percent rows successfully removed


---
## Cell 7 — Standardise State Names and Separate India Row

Two operations:
1. **Apply DISEASE_STATE_CLEAN_MAP** — defensively corrects any variant spellings
2. **Separate `India` row** — national totals must never be averaged with state rows

After this cell, we also cross-check state names against Phase 1's output to confirm the Phase 3 join key is perfectly aligned.

In [7]:
# ── Apply state name standardisation ──────────────────────────────────────────
# .map() replaces values found in the dict.
# .fillna(df_disease['location_name']) means: if a state name is NOT in the map,
# keep its original value (do not replace it with NaN)
df_disease['location_name'] = (
    df_disease['location_name']
    .map(DISEASE_STATE_CLEAN_MAP)
    .fillna(df_disease['location_name'])
)

# ── Separate national aggregate (India) from state rows ───────────────────────
# mask_national = True for rows where location_name is 'India' etc.
# ~mask_national = True for rows where location_name is NOT 'India' etc.
mask_national  = df_disease['location_name'].isin(NATIONAL_AGGREGATE_NAMES)
df_national    = df_disease[mask_national].copy()
df_states      = df_disease[~mask_national].copy()

print(f"National aggregate rows : {len(df_national):,}  (India — will be saved separately)")
print(f"State-level rows        : {len(df_states):,}")
print()

# ── Print all state names in disease data ─────────────────────────────────────
states_in_disease = sorted(df_states['location_name'].unique())
print(f"States in disease data ({len(states_in_disease)} total):")
for s in states_in_disease:
    print(f"   {s}")
print()

# ── Cross-check against Phase 1 standardised names ───────────────────────────
# WHY: If any name doesn't match, Phase 3's join will silently drop that state.
# We catch the problem NOW, not after Phase 3 produces incomplete results.
#
# These are the state names Phase 1 saved after applying STATE_NAME_MAP:
POLLUTION_STATES_AFTER_PHASE1 = {
    'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar',
    'Other Union Territories',       # Chandigarh was mapped here in Phase 1
    'Chhattisgarh', 'Delhi', 'Gujarat', 'Haryana', 'Himachal Pradesh',
    'Jammu & Kashmir and Ladakh',    # 'Jammu and Kashmir' was mapped here in Phase 1
    'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra',
    'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha',
    'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana',
    'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal'
    # NOTE: Puducherry was also mapped to 'Other Union Territories' in Phase 1
}

disease_set           = set(states_in_disease)
only_in_disease       = disease_set - POLLUTION_STATES_AFTER_PHASE1
only_in_pollution     = POLLUTION_STATES_AFTER_PHASE1 - disease_set
perfectly_matched     = disease_set & POLLUTION_STATES_AFTER_PHASE1

print(f"States matching perfectly (Phase 3 join will work for these): {len(perfectly_matched)}")
print()

if only_in_disease:
    print("States in disease data with NO matching pollution data:")
    for s in sorted(only_in_disease):
        print(f"   '{s}' — will appear in disease_master but get NaN pollution values in Phase 3")
    print()
else:
    print("No disease states are absent from pollution data")
    print()

if only_in_pollution:
    print("Pollution states with NO matching disease data:")
    for s in sorted(only_in_pollution):
        print(f"   '{s}'")
else:
    print("No pollution states are absent from disease data")

National aggregate rows : 588  (India — will be saved separately)
State-level rows        : 18,228

States in disease data (31 total):
   Andhra Pradesh
   Arunachal Pradesh
   Assam
   Bihar
   Chhattisgarh
   Delhi
   Goa
   Gujarat
   Haryana
   Himachal Pradesh
   Jammu & Kashmir and Ladakh
   Jharkhand
   Karnataka
   Kerala
   Madhya Pradesh
   Maharashtra
   Manipur
   Meghalaya
   Mizoram
   Nagaland
   Odisha
   Other Union Territories
   Punjab
   Rajasthan
   Sikkim
   Tamil Nadu
   Telangana
   Tripura
   Uttar Pradesh
   Uttarakhand
   West Bengal

States matching perfectly (Phase 3 join will work for these): 30

States in disease data with NO matching pollution data:
   'Goa' — will appear in disease_master but get NaN pollution values in Phase 3

No pollution states are absent from disease data


---
## Cell 8 — Add Evidence Strength Column

We map every cause name to `direct`, `strong`, or `indirect`.  

Because the cause names were confirmed from Phase 0.5 Cell 13, **we expect zero `unknown` rows**.  
If any appear, the cell prints the exact unmatched string so you can fix it in Cell 3.

In [9]:
# ── Map cause_name → evidence_strength ───────────────────────────────────────
# .map() looks up each cause_name value in EVIDENCE_STRENGTH_MAP
# .fillna('unknown') catches any cause that was not found in the map
# WHY 'unknown' and not NaN: 'unknown' is an obvious flag in the CSV;
# NaN rows are easy to overlook and cause silent errors later
df_states['evidence_strength'] = (
    df_states['cause_name']
    .map(EVIDENCE_STRENGTH_MAP)
    .fillna('unknown')
)

# ── Print distribution ────────────────────────────────────────────────────────
print("Evidence strength distribution:")
for label, count in df_states['evidence_strength'].value_counts().items():
    pct = count / len(df_states) * 100
    print(f"   {label:12s}: {count:,} rows ({pct:.1f}%)")

print()

# ── Flag any unmatched cause names ───────────────────────────────────────────
unknown_causes = (
    df_states[df_states['evidence_strength'] == 'unknown']['cause_name'].unique()
)

if len(unknown_causes) > 0:
    print(f"WARNING: {len(unknown_causes)} cause name(s) not matched in EVIDENCE_STRENGTH_MAP:")
    for cause in sorted(unknown_causes):
        print(f"   EXACT STRING IN FILE: '{cause}'")
    print()
    print("   FIX: Go to Cell 3 and add these exact strings to EVIDENCE_STRENGTH_MAP.")
    print("        Copy the string character-for-character including capitals.")
else:
    print("All 21 cause names matched — zero unknown rows (as expected)")

Evidence strength distribution:
   direct      : 11,284 rows (61.9%)
   strong      : 4,340 rows (23.8%)
   indirect    : 2,604 rows (14.3%)

All 21 cause names matched — zero unknown rows (as expected)


---
## Cell 9 — Select Final Columns and Sort

We keep only the columns needed for Phase 3 analysis and discard the rest.

| Column | What it contains | Why we keep it |
|---|---|---|
| `location_name` | State name | Phase 3 join key |
| `year` | 2010 – 2023 | Phase 3 join key |
| `cause_name` | Disease name (e.g. COPD) | Analysis |
| `metric_id` | 1=Number, 3=Rate | Filtering in Phase 3 |
| `metric_name` | 'Number' or 'Rate' | Human-readable label |
| `val` | Death count or deaths per 100k | Core analysis value |
| `upper` | Upper uncertainty bound | Confidence reporting |
| `lower` | Lower uncertainty bound | Confidence reporting |
| `evidence_strength` | direct / strong / indirect | Analytical honesty |
| `source_file` | Which of the 5 CSVs this row came from | Debugging |

In [11]:
# ── Desired final column order ────────────────────────────────────────────────
FINAL_COLUMNS = [
    'location_name',
    'year',
    'cause_name',
    'metric_id',
    'metric_name',
    'val',
    'upper',
    'lower',
    'evidence_strength',
    'source_file',
]

# ── Keep only columns that exist in the DataFrame ────────────────────────────
# WHY: 'upper' and 'lower' might not exist in all GBD export versions.
# Rather than crashing, we skip them silently and report which were skipped.
cols_present = [c for c in FINAL_COLUMNS if c in df_states.columns]
cols_missing = [c for c in FINAL_COLUMNS if c not in df_states.columns]

if cols_missing:
    print(f"Note: these expected columns were not found and will be skipped: {cols_missing}")
else:
    print("All expected columns present")

# ── Select and sort ───────────────────────────────────────────────────────────
df_final = df_states[cols_present].copy()

# Sort so the CSV is easy to read: state → year → disease → metric type
df_final.sort_values(
    ['location_name', 'year', 'cause_name', 'metric_id'],
    inplace=True
)
df_final.reset_index(drop=True, inplace=True)

print(f"\nFinal state-level DataFrame: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print()
print("First 4 rows:")
print(df_final.head(4).to_string())

All expected columns present

Final state-level DataFrame: 18,228 rows x 10 columns

First 4 rows:
    location_name  year                       cause_name  metric_id metric_name          val        upper        lower evidence_strength                  source_file
0  Andhra Pradesh  2010                           Asthma          1      Number  5470.123668  9851.214143  3001.040867            direct  airpollutiondiseases2nd.csv
1  Andhra Pradesh  2010                           Asthma          3        Rate    10.739414    19.340745     5.891900            direct  airpollutiondiseases2nd.csv
2  Andhra Pradesh  2010  Atrial fibrillation and flutter          1      Number  1034.536609  1413.395932   763.100787            direct  airpollutiondiseases3rd.csv
3  Andhra Pradesh  2010  Atrial fibrillation and flutter          3        Rate     2.031091     2.774900     1.498185            direct  airpollutiondiseases3rd.csv


---
## Cell 10 — Quality Checks

Six checks before saving.  
Expected values are hardcoded from Phase 0.5 — any deviation is a real problem worth investigating.

In [12]:
print("=" * 60)
print("PHASE 2 QUALITY CHECKS")
print("=" * 60)

# ── Check 1: Year range ───────────────────────────────────────────────────────
print("\n[CHECK 1] Year range")
years = sorted(df_final['year'].unique())
print(f"   Years present: {years[0]} to {years[-1]}  ({len(years)} years)")
if len(years) == 14 and years[0] == 2010 and years[-1] == 2023:
    print("   OK: All 14 years present (2010-2023)")
else:
    missing_yrs = sorted(set(range(2010, 2024)) - set(years))
    print(f"   WARNING: Expected 14 years. Missing: {missing_yrs}")

# ── Check 2: NaN in key columns ──────────────────────────────────────────────
print("\n[CHECK 2] NaN counts in key columns (all should be 0)")
for col in ['location_name', 'cause_name', 'year', 'val', 'evidence_strength']:
    if col not in df_final.columns:
        continue
    n_nan  = df_final[col].isna().sum()
    status = "OK" if n_nan == 0 else "WARNING"
    print(f"   [{status}]  {col:25s}: {n_nan:,} NaN values")

# ── Check 3: State count ─────────────────────────────────────────────────────
print("\n[CHECK 3] State count")
n_states = df_final['location_name'].nunique()
print(f"   States in disease_master: {n_states}")
print(f"   (Expected ~32: 31 pollution states + Goa which has no pollution stations)")
if n_states < 28:
    print("   WARNING: Fewer states than expected — check Cell 7")

# ── Check 4: No Percent metric rows ──────────────────────────────────────────
print("\n[CHECK 4] Metric types remaining (should be Number + Rate only)")
metrics = df_final.groupby(['metric_id', 'metric_name']).size().reset_index(name='rows')
for _, row in metrics.iterrows():
    flag = "WARNING — should have been dropped!" if row['metric_id'] == 2 else "OK"
    print(f"   metric_id={int(row['metric_id'])} ({row['metric_name']}): {row['rows']:,} rows  [{flag}]")

# ── Check 5: Evidence strength — no unknown rows ──────────────────────────────
print("\n[CHECK 5] Evidence strength (should have zero 'unknown' rows)")
ev_counts = df_final['evidence_strength'].value_counts()
for label, count in ev_counts.items():
    flag = "WARNING — fix EVIDENCE_STRENGTH_MAP" if label == 'unknown' else "OK"
    print(f"   {label:12s}: {count:,} rows  [{flag}]")

# ── Check 6: Total row count ──────────────────────────────────────────────────
print("\n[CHECK 6] Total row count")
print(f"   Actual:   {len(df_final):,} rows")
print(f"   Expected: approximately 18,816 rows")
print(f"   Formula:  28,224 raw rows × (2/3) after removing Percent metric")
print(f"             minus however many 'India' national rows existed in each file")

print()
print("=" * 60)

PHASE 2 QUALITY CHECKS

[CHECK 1] Year range
   Years present: 2010 to 2023  (14 years)
   OK: All 14 years present (2010-2023)

[CHECK 2] NaN counts in key columns (all should be 0)
   [OK]  location_name            : 0 NaN values
   [OK]  cause_name               : 0 NaN values
   [OK]  year                     : 0 NaN values
   [OK]  val                      : 0 NaN values
   [OK]  evidence_strength        : 0 NaN values

[CHECK 3] State count
   States in disease_master: 31
   (Expected ~32: 31 pollution states + Goa which has no pollution stations)

[CHECK 4] Metric types remaining (should be Number + Rate only)
   metric_id=1 (Number): 9,114 rows  [OK]
   metric_id=3 (Rate): 9,114 rows  [OK]

[CHECK 5] Evidence strength (should have zero 'unknown' rows)
   direct      : 11,284 rows  [OK]
   strong      : 4,340 rows  [OK]
   indirect    : 2,604 rows  [OK]

[CHECK 6] Total row count
   Actual:   18,228 rows
   Expected: approximately 18,816 rows
   Formula:  28,224 raw rows × (2/3)

---
## Cell 11 — Save Output Files

In [14]:
# ── Save state-level disease master ──────────────────────────────────────────
disease_master_path = OUTPUT_DIR / 'disease_master.csv'
df_final.to_csv(disease_master_path, index=False)

print(f"disease_master.csv saved")
print(f"   Path  : {disease_master_path}")
print(f"   Shape : {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print()

# ── Save national aggregate rows separately ───────────────────────────────────
# WHY: df_national was separated in Cell 7 — BEFORE Cell 8 added evidence_strength.
# So we cannot use cols_present (which includes evidence_strength) on df_national.
# We build the column list fresh using only what actually exists in df_national.

df_national_out = df_national[
    df_national['metric_id'].isin([1, 3])
].copy()

# Add evidence_strength to national rows the same way Cell 8 did for state rows
df_national_out['evidence_strength'] = (
    df_national_out['cause_name']
    .map(EVIDENCE_STRENGTH_MAP)
    .fillna('unknown')
)

# Now select only columns that exist in this DataFrame
national_cols = [c for c in cols_present if c in df_national_out.columns]

df_national_out = df_national_out[national_cols].copy()
df_national_out.sort_values(
    ['location_name', 'year', 'cause_name', 'metric_id'],
    inplace=True
)

national_path = OUTPUT_DIR / 'disease_national.csv'
df_national_out.to_csv(national_path, index=False)

print(f"disease_national.csv saved  (India national totals — do NOT join with state pollution data)")
print(f"   Path  : {national_path}")
print(f"   Shape : {df_national_out.shape[0]:,} rows")

disease_master.csv saved
   Path  : ..\outputs\disease_master.csv
   Shape : 18,228 rows x 10 columns

disease_national.csv saved  (India national totals — do NOT join with state pollution data)
   Path  : ..\outputs\disease_national.csv
   Shape : 588 rows


---
## Cell 12 — Phase 2 Summary and Next Steps

In [15]:
print("=" * 60)
print("PHASE 2 COMPLETE — SUMMARY")
print("=" * 60)
print()

print("Files saved:")
print(f"  outputs/disease_master.csv")
print(f"      {df_final.shape[0]:,} rows — state-level deaths, 2010-2023")
print(f"  outputs/disease_national.csv")
print(f"      {df_national_out.shape[0]:,} rows — India national totals (never mix with states)")
print()

print("Columns in disease_master.csv:")
for col in df_final.columns:
    sample_vals = df_final[col].dropna().iloc[:2].tolist()
    print(f"  {col:25s}  e.g. {sample_vals}")
print()

print("Key decisions made in Phase 2:")
print("  Kept  : metric_id 1 (Number) and 3 (Rate)")
print("  Dropped: metric_id 2 (Percent) — not useful for death correlation")
print("  India national row saved separately — never mixed into state analysis")
print("  State names already in GBD format — confirmed matching Phase 1 output")
print("  evidence_strength added to every row")
print("  source_file retained for traceability back to original CSV")
print()

print("-" * 60)
print("NEXT: Phase 3 — Join Pollution + Disease (phase_3_join.ipynb)")
print()
print("  Phase 3 will:")
print("    1. Load outputs/air_pollution_master.csv  (city x month)")
print("    2. Aggregate: city-monthly  →  state-annual pollution averages")
print("    3. Load outputs/disease_master.csv  (state x year)")
print("    4. Join the two on: state name + year")
print("    5. Save outputs/combined_master.csv  (the main analysis table)")
print()
print("  Join keys (must match exactly between both files):")
print("    Pollution column : 'state'          (standardised in Phase 1)")
print("    Disease column   : 'location_name'  (standardised in Phase 2)")
print("    Both now use identical GBD state names.")
print("=" * 60)

PHASE 2 COMPLETE — SUMMARY

Files saved:
  outputs/disease_master.csv
      18,228 rows — state-level deaths, 2010-2023
  outputs/disease_national.csv
      588 rows — India national totals (never mix with states)

Columns in disease_master.csv:
  location_name              e.g. ['Andhra Pradesh', 'Andhra Pradesh']
  year                       e.g. [2010, 2010]
  cause_name                 e.g. ['Asthma', 'Asthma']
  metric_id                  e.g. [1, 3]
  metric_name                e.g. ['Number', 'Rate']
  val                        e.g. [5470.123668188653, 10.73941387443099]
  upper                      e.g. [9851.214142579767, 19.340744791213236]
  lower                      e.g. [3001.040867400543, 5.891899687117691]
  evidence_strength          e.g. ['direct', 'direct']
  source_file                e.g. ['airpollutiondiseases2nd.csv', 'airpollutiondiseases2nd.csv']

Key decisions made in Phase 2:
  Kept  : metric_id 1 (Number) and 3 (Rate)
  Dropped: metric_id 2 (Percent) — not 